# MeguvizCAC v0.3-Loop — Extensible Byte-Fallback + Prelude + RRT Core Loop

Implements locked **Issue #5** architecture:
- **Zero-`[UNK]` Extensible Embedding**: Full `21,128` WordPiece Weighted Truncated SVD (`E=128/192`) + `+256` UTF-8 Byte-Fallback slots (`21,384` total)
- **Huginn + RRT Recurrent Core**: `Prelude(1) + Core(core_layers x T)` with SVD LoRA init, empirical QK-Norm, Gated Input Injection, and Ouro entropy-regularized adaptive exit
- **Zero-Duplication Dual Static ONNX + INT8 QAT**: `prelude.onnx` + `core_step.onnx` + length-sorted compact-batch engine + optional `W4-Emb / W8-Body` opset-21 Edge bundle
- **GPU Auto-Tuning**: `--batch-size 0` selects `256` on A100 (>=35 GB VRAM) or `128` on L4, with TF32 + BF16 + LengthBucketBatchSampler


## 1. GPU Check & Repo Setup


In [ ]:
!nvidia-smi
import os, subprocess, torch
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3) if torch.cuda.is_available() else 0.0
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"Detected GPU: {gpu_name} | VRAM: {vram_gb:.1f} GB | bf16: {torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False}")

REPO_DIR = "/content/MeguvizCAC"
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/jichuo1/MeguvizCAC.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only
%cd /content/MeguvizCAC
!git log -1 --oneline
!pip -q install -U onnx onnxruntime transformers scipy scikit-learn


## 2. Download Stage-1 (`v0.1-data`) & Stage-2 (`v0.2-data`) Releases + Dry Run


In [ ]:
!mkdir -p /content/data/stage2
!curl -fL "https://github.com/jichuo1/MeguvizCAC/releases/download/v0.1-data/bili-hydrogen-rescored-350387.jsonl.gz" -o /content/data/bili-hydrogen-rescored-350387.jsonl.gz
!curl -fL "https://github.com/jichuo1/MeguvizCAC/releases/download/v0.2-data/stage2_distill_bundle.zip" -o /content/data/stage2_distill_bundle.zip
!unzip -qo /content/data/stage2_distill_bundle.zip -d /content/data/stage2
!ls -lh /content/data /content/data/stage2

!python -m meguviz_cac.train_v03_loop --dry-run \
  --data /content/data/bili-hydrogen-rescored-350387.jsonl.gz \
  --qwen-data /content/data/stage2/stage2-fused-train.jsonl.gz \
  --pairs-data /content/data/stage2/stage2-literary-pairs.jsonl


## 3. 60-Second End-to-End Smoke Test (Train + QAT + Exit Policy + Dual ONNX + W4 Edge Check)


In [ ]:
!python -m meguviz_cac.train_v03_loop --smoke-test \
  --data /content/data/bili-hydrogen-rescored-350387.jsonl.gz \
  --qwen-data /content/data/stage2/stage2-fused-train.jsonl.gz \
  --pairs-data /content/data/stage2/stage2-literary-pairs.jsonl \
  --version v0.3-smoke \
  --out /content/out_v03_smoke


## 4. Ablation 1 (Primary Target): `1 + 2x4` RRT Core, `E=128`, `r=16`


In [ ]:
!python -m meguviz_cac.train_v03_loop \
  --data /content/data/bili-hydrogen-rescored-350387.jsonl.gz \
  --qwen-data /content/data/stage2/stage2-fused-train.jsonl.gz \
  --pairs-data /content/data/stage2/stage2-literary-pairs.jsonl \
  --version v0.3-loop-2x4-e128 \
  --embed-dim 128 --core-layers 2 --num-loops 4 --normal-max-loops 3 --lora-rank 16 \
  --epochs 4 --qat-epochs 1 --batch-size 0 --synth-cap 5000 \
  --out /content/out_v03_2x4_e128


## 5. Ablation 2 (Higher-Capacity Vocab): `1 + 2x4` RRT Core, `E=192`, `r=16`


In [ ]:
!python -m meguviz_cac.train_v03_loop \
  --data /content/data/bili-hydrogen-rescored-350387.jsonl.gz \
  --qwen-data /content/data/stage2/stage2-fused-train.jsonl.gz \
  --pairs-data /content/data/stage2/stage2-literary-pairs.jsonl \
  --version v0.3-loop-2x4-e192 \
  --embed-dim 192 --core-layers 2 --num-loops 4 --normal-max-loops 3 --lora-rank 16 \
  --epochs 4 --qat-epochs 1 --batch-size 0 --synth-cap 5000 \
  --out /content/out_v03_2x4_e192


## 6. Ablation 3 (Ultralight): `1 + 1x5` RRT Core, `E=128`, `r=16`


In [ ]:
!python -m meguviz_cac.train_v03_loop \
  --data /content/data/bili-hydrogen-rescored-350387.jsonl.gz \
  --qwen-data /content/data/stage2/stage2-fused-train.jsonl.gz \
  --pairs-data /content/data/stage2/stage2-literary-pairs.jsonl \
  --version v0.3-loop-1x5-e128 \
  --embed-dim 128 --core-layers 1 --num-loops 5 --normal-max-loops 4 --lora-rank 16 \
  --epochs 4 --qat-epochs 1 --batch-size 0 --synth-cap 5000 \
  --out /content/out_v03_1x5_e128


## 7. Apples-to-Apples Comparison (`v0.1` vs `v0.2` vs `v0.3-Loop`), Adversarial Runtime Stress & Bundle Export


In [ ]:
import json, subprocess, zipfile
from pathlib import Path

baselines = [
    ("v0.1 (4L bge-small INT8)", 0.9552, 0.8537, 0.8667, 0.7500, 23.94, 4.0, 9.73),
    ("v0.2 (4L Dual-Teacher INT8)", 0.9530, 0.8598, 0.9167, 0.8333, 23.94, 4.0, 9.86),
]

run_dirs = [
    Path("/content/out_v03_2x4_e128"),
    Path("/content/out_v03_2x4_e192"),
    Path("/content/out_v03_1x5_e128"),
]

print(f"{'Model':32s} | {'test_auroc':10s} | {'test_sp':7s} | {'probe_auc':9s} | {'probe_r70':9s} | {'ONNX MB':7s} | {'AvgStep':7s} | {'CPU ms (1 / b32)':16s}")
print("-" * 116)
for name, ta, ts, pa, pr, mb, st, ms in baselines:
    print(f"{name:32s} | {ta:10.4f} | {ts:7.4f} | {pa:9.4f} | {pr:9.4f} | {mb:7.2f} | {st:7.2f} | {ms:.2f} / -")

for d in run_dirs:
    mf = list(d.glob("metrics-*.json"))
    if not mf:
        continue
    m = json.loads(mf[0].read_text(encoding="utf-8"))
    tb = m["test_bili_gold"]
    prb = m["probe_u001_u017"]
    ox = m["onnx_int8"]
    print(
        f"{m['version']:32s} | "
        f"{tb.get('auroc', 0.0):10.4f} | "
        f"{tb['spearman']:7.4f} | "
        f"{prb.get('auroc', 0.0):9.4f} | "
        f"{prb['at_070']['recall']:9.4f} | "
        f"{ox['total_onnx_bytes'] / (1024**2):7.2f} | "
        f"{tb.get('avg_steps', 0.0):7.2f} | "
        f"{ox['single_1thread_ms']:.2f} / {ox['compact_batch32_1thread_ms']:.2f}"
    )

# Run standardized adversarial + zero-[UNK] verification on primary zip
primary_zip = Path("/content/out_v03_2x4_e128/meguvizcac-v0.3-loop-2x4-e128-int8.zip")
if primary_zip.is_file():
    print("\n=== v03_runtime Stress & Zero-[UNK] Verification ===")
    !python -m meguviz_cac.v03_runtime --bundle {primary_zip}

# Package completed outputs for fast download
bundle_out = Path("/content/v03_ablation_bundles.zip")
with zipfile.ZipFile(bundle_out, "w", compression=zipfile.ZIP_STORED) as zf:
    for d in run_dirs:
        for pat in ("meguvizcac-*.zip", "metrics-*.json", "manifest.json"):
            for fp in d.glob(pat):
                zf.write(fp, f"{d.name}/{fp.name}")
print(f"Packaged downloadable bundles -> {bundle_out} ({bundle_out.stat().st_size / (1024**2):.2f} MB)")
